## 1. Установка зависимостей


In [1]:

import subprocess
pkgs = ["sentence-transformers", "chromadb", "pydantic", "faker", "numpy", "tqdm"]
for pkg in pkgs:
    subprocess.run(["pip", "install", pkg, "-q"], check=True)
print("Все зависимости установлены.")


Все зависимости установлены.


## 2. Генерация синтетического датасета из 150 отелей


In [2]:

import json, random, uuid
from faker import Faker

fake = Faker("ru_RU")
random.seed(42)

CITIES = {
    "Сочи": {"lat": (43.55, 43.65), "lon": (39.7, 39.8), "areas": ["морской берег", "центр", "горный курорт"]},
    "Красная Поляна": {"lat": (43.65, 43.72), "lon": (40.15, 40.25), "areas": ["горный курорт", "горы"]},
    "Санкт-Петербург": {"lat": (59.90, 59.98), "lon": (30.2, 30.4), "areas": ["центр", "исторический район", "набережная"]},
    "Москва": {"lat": (55.70, 55.80), "lon": (37.5, 37.7), "areas": ["центр", "деловой район", "парковая зона"]},
    "Алтай": {"lat": (51.8, 52.1), "lon": (85.5, 86.0), "areas": ["горы", "горный курорт", "природный парк"]},
    "Казань": {"lat": (55.75, 55.82), "lon": (49.1, 49.2), "areas": ["центр", "исторический район", "речная набережная"]},
}

AREA_TYPE_MAP = {
    "морской берег": "sea", "горный курорт": "mountains", "горы": "mountains",
    "центр": "center", "исторический район": "center", "набережная": "sea",
    "деловой район": "center", "парковая зона": "center",
    "природный парк": "mountains", "речная набережная": "sea",
}

VIBES = {
    "mountains": ["уютная горная атмосфера с видом на заснеженные вершины",
                  "идеально для активного отдыха и трекинга в горах",
                  "панорамные виды на горные хребты, тишина и чистый воздух"],
    "sea": ["свежий морской бриз и шум прибоя за окном",
            "пляж в 5 минутах ходьбы, закаты над морем",
            "уютный отель у моря с видом на набережную"],
    "center": ["в самом сердце города, рядом со всеми достопримечательностями",
               "современный городской отель с отличной транспортной доступностью",
               "шаговая доступность ресторанов, музеев и торговых центров"],
}

HOTEL_PREFIXES = ["Гранд", "Уютный", "Лесной", "Горный", "Морской", "Премиум", "Камерный", "Панорама"]
HOTEL_SUFFIXES = ["Отель", "Резорт", "Хаус", "Вилла", "Суит", "Инн", "Лодж", "Апарт"]

def gen_hotel(idx, city, city_data):
    area = random.choice(city_data["areas"])
    area_type = AREA_TYPE_MAP[area]
    lat = round(random.uniform(*city_data["lat"]), 6)
    lon = round(random.uniform(*city_data["lon"]), 6)

    pet_friendly = random.random() < 0.35
    heated_pool = random.random() < 0.4
    spa = random.random() < 0.45
    fast_wifi = random.random() < 0.85
    parking = random.random() < 0.6
    breakfast = random.random() < 0.5

    price = random.randint(3500, 35000)
    rating = round(random.uniform(3.2, 5.0), 1)
    reviews = random.randint(12, 1800)

    vibe = random.choice(VIBES[area_type])
    amenity_desc = []
    if heated_pool: amenity_desc.append("подогреваемый бассейн")
    if spa: amenity_desc.append("спа-центр")
    if pet_friendly: amenity_desc.append("размещение с питомцами")
    if breakfast: amenity_desc.append("завтрак включён")
    if parking: amenity_desc.append("бесплатная парковка")

    description = (
        f"{vibe}. "
        f"Отель расположен в районе «{area}» города {city}. "
        + (f"Предлагает: {', '.join(amenity_desc)}. " if amenity_desc else "")
        + f"Цена от {price:,} ₽/ночь. Рейтинг гостей: {rating}/5.0."
    )

    min_stay = random.choice([1, 1, 1, 2, 2, 3])
    max_pets = random.choice([1, 2]) if pet_friendly else 0

    return {
        "id": str(uuid.uuid4()),
        "name": f"{random.choice(HOTEL_PREFIXES)} {random.choice(HOTEL_SUFFIXES)} {idx}",
        "city": city,
        "area": area,
        "area_type": area_type,
        "price_per_night_rub": price,
        "rating": rating,
        "reviews_count": reviews,
        "amenities": {
            "pet_friendly": pet_friendly,
            "heated_pool": heated_pool,
            "spa": spa,
            "fast_wifi": fast_wifi,
            "parking": parking,
            "breakfast_included": breakfast,
        },
        "rules": {
            "min_stay_nights": min_stay,
            "max_pets_allowed": max_pets,
            "pets_deposit_rub": random.randint(500, 3000) if pet_friendly else 0,
        },
        "description": description,
        "coordinates": {"lat": lat, "lon": lon},
        "image_urls": [
            f"https://images.ostrovok.ru/mock/{uuid.uuid4().hex[:8]}_room{i}.jpg"
            for i in range(1, 4)
        ],
    }

hotels = []
city_list = list(CITIES.keys())
per_city = [25, 25, 30, 25, 20, 25]  # итого 150

idx = 1
for city, count in zip(city_list, per_city):
    for _ in range(count):
        hotels.append(gen_hotel(idx, city, CITIES[city]))
        idx += 1

import os
os.makedirs("./outputs", exist_ok=True)
with open("./outputs/hotels_dataset.json", "w", encoding="utf-8") as f:
    json.dump(hotels, f, ensure_ascii=False, indent=2)

print(f"✅ Датасет сгенерирован: {len(hotels)} отелей")
print(f"   Городá: {set(h['city'] for h in hotels)}")
print(f"   Pet-friendly: {sum(h['amenities']['pet_friendly'] for h in hotels)}")
print(f"   С бассейном: {sum(h['amenities']['heated_pool'] for h in hotels)}")


✅ Датасет сгенерирован: 150 отелей

   Городá: {'Москва', 'Санкт-Петербург', 'Алтай', 'Сочи', 'Казань', 'Красная Поляна'}

   Pet-friendly: 61

   С бассейном: 56


## 3. Генерация эмбеддингов и построение Chroma-индекса


In [3]:

import chromadb
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

chroma_client = chromadb.PersistentClient(path="./outputs/chroma_db")

try:
    chroma_client.delete_collection("hotels")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="hotels",
    metadata={"hnsw:space": "cosine"}
)

descriptions = [h["description"] for h in hotels]
print("Генерация эмбеддингов...")
embeddings = model.encode(descriptions, batch_size=32, show_progress_bar=True).tolist()

ids = [h["id"] for h in hotels]
metadatas = [
    {
        "name": h["name"],
        "city": h["city"],
        "area_type": h["area_type"],
        "price_per_night_rub": h["price_per_night_rub"],
        "rating": h["rating"],
        "reviews_count": h["reviews_count"],
        "pet_friendly": str(h["amenities"]["pet_friendly"]),
        "heated_pool": str(h["amenities"]["heated_pool"]),
        "spa": str(h["amenities"]["spa"]),
        "fast_wifi": str(h["amenities"]["fast_wifi"]),
        "parking": str(h["amenities"]["parking"]),
        "breakfast_included": str(h["amenities"]["breakfast_included"]),
        "min_stay_nights": h["rules"]["min_stay_nights"],
        "max_pets_allowed": h["rules"]["max_pets_allowed"],
        "lat": h["coordinates"]["lat"],
        "lon": h["coordinates"]["lon"],
    }
    for h in hotels
]

collection.add(ids=ids, embeddings=embeddings, documents=descriptions, metadatas=metadatas)
print(f"✅ Chroma-индекс создан: {collection.count()} документов")


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Генерация эмбеддингов...


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

✅ Chroma-индекс создан: 150 документов


## 4. Модуль поиска: Pydantic-схемы + функция `search_hotels`


In [7]:

search_module_code = '''"""
hotel_search.py — Tool Interface для LangGraph / FastAPI
Использование:
    from hotel_search import search_hotels, HotelFilterSchema
"""
from typing import Optional, List
from pydantic import BaseModel, Field
import chromadb
from sentence_transformers import SentenceTransformer
import asyncio

class HotelFilterSchema(BaseModel):
    city: Optional[str] = Field(None, description="Город (напр. 'Сочи')")
    area_type: Optional[str] = Field(None, description="mountains | sea | center")
    max_price_per_night_rub: Optional[int] = Field(None, description="Макс. цена за ночь, ₽")
    min_rating: Optional[float] = Field(None, description="Мин. рейтинг (1.0–5.0)")
    pet_friendly: Optional[bool] = Field(None, description="Разрешены питомцы")
    heated_pool: Optional[bool] = Field(None, description="Подогреваемый бассейн")
    spa: Optional[bool] = Field(None, description="Спа-центр")
    breakfast_included: Optional[bool] = Field(None, description="Завтрак включён")
    top_k: int = Field(5, description="Количество результатов")


class HotelResult(BaseModel):
    id: str
    name: str
    city: str
    area_type: str
    price_per_night_rub: int
    rating: float
    reviews_count: int
    pet_friendly: bool
    heated_pool: bool
    spa: bool
    description: str
    score: float = Field(..., description="Косинусное сходство (0–1)")


_model = None
_collection = None

def _init():
    global _model, _collection
    if _model is None:
        _model = SentenceTransformer(
            "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
        )
    if _collection is None:
        client = chromadb.PersistentClient(path="./outputs/chroma_db")
        _collection = client.get_collection("hotels")


async def search_hotels(
    filters: HotelFilterSchema,
    semantic_query: str,
) -> List[HotelResult]:
    """
    Гибридный поиск: семантический (cosine) + хард-фильтры по метаданным.
    """
    _init()

    where_clauses = []
    if filters.city:
        where_clauses.append({"city": {"$eq": filters.city}})
    if filters.area_type:
        where_clauses.append({"area_type": {"$eq": filters.area_type}})
    if filters.max_price_per_night_rub:
        where_clauses.append({"price_per_night_rub": {"$lte": filters.max_price_per_night_rub}})
    if filters.min_rating:
        where_clauses.append({"rating": {"$gte": filters.min_rating}})
    if filters.pet_friendly is not None:
        where_clauses.append({"pet_friendly": {"$eq": str(filters.pet_friendly)}})
    if filters.heated_pool is not None:
        where_clauses.append({"heated_pool": {"$eq": str(filters.heated_pool)}})
    if filters.spa is not None:
        where_clauses.append({"spa": {"$eq": str(filters.spa)}})
    if filters.breakfast_included is not None:
        where_clauses.append({"breakfast_included": {"$eq": str(filters.breakfast_included)}})

    where = {"$and": where_clauses} if len(where_clauses) > 1 else (where_clauses[0] if where_clauses else None)

    query_embedding = await asyncio.to_thread(_model.encode, [semantic_query])

    query_kwargs = dict(
        query_embeddings=query_embedding.tolist(),
        n_results=min(filters.top_k, _collection.count()),
        include=["metadatas", "documents", "distances"],
    )
    if where:
        query_kwargs["where"] = where

    results = _collection.query(**query_kwargs)

    hotels_out = []
    for i, (meta, doc, dist) in enumerate(zip(
        results["metadatas"][0], results["documents"][0], results["distances"][0]
    )):
        hotels_out.append(HotelResult(
            id=results["ids"][0][i],
            name=meta["name"],
            city=meta["city"],
            area_type=meta["area_type"],
            price_per_night_rub=int(meta["price_per_night_rub"]),
            rating=float(meta["rating"]),
            reviews_count=int(meta["reviews_count"]),
            pet_friendly=meta["pet_friendly"] == "True",
            heated_pool=meta["heated_pool"] == "True",
            spa=meta["spa"] == "True",
            description=doc,
            score=round(1 - dist, 4),
        ))
    return hotels_out
'''

with open("./hotel_search.py", "w", encoding="utf-8") as f:
    f.write(search_module_code)

print("✅ Модуль hotel_search.py обновлён")


✅ Модуль hotel_search.py обновлён


## 5. Тест end-to-end: запрос «уикенд в Сочи с собакой, бассейн, горы, бюджет 15000₽»


In [9]:

import importlib.util, asyncio, pandas as pd
from IPython.display import display

spec = importlib.util.spec_from_file_location("hotel_search", "./hotel_search.py")
hs = importlib.util.module_from_spec(spec)
spec.loader.exec_module(hs)

filters = hs.HotelFilterSchema(
    city="Сочи",
    pet_friendly=True,
    heated_pool=True,
    max_price_per_night_rub=15000,
    top_k=5,
)

results = await hs.search_hotels(
    filters=filters,
    semantic_query="уютный горный отдых с видом на вершины, тихо, для двоих с собакой"
)

df = pd.DataFrame([r.model_dump() for r in results])[
    ["name", "city", "area_type", "price_per_night_rub", "rating", "pet_friendly", "heated_pool", "score", "description"]
]
display(df, metadata={"alt": "Результаты поиска отелей"})


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

,name,city,area_type,price_per_night_rub,rating,pet_friendly,heated_pool,score,description
0,Горный Лодж 1,Сочи,mountains,4541,3.3,True,True,0.5066,уютная горная атмосфера с видом на заснеженные...
1,Уютный Лодж 3,Сочи,mountains,5779,3.3,True,True,0.3710,идеально для активного отдыха и трекинга в гор...
2,Морской Отель 20,Сочи,sea,11728,4.8,True,True,0.3208,свежий морской бриз и шум прибоя за окном. Оте...
3,Премиум Резорт 18,Сочи,center,6776,3.3,True,True,0.2877,"шаговая доступность ресторанов, музеев и торго..."


## Скачать проект через Camber CLI (из ноутбука)


In [10]:
camber version --api-key $CAMBER_TOKEN

Camber 1.0.37 - e8a8e29


## 6. Генерация расширенного датасета — 200 отелей (полная схема)


In [11]:

import json, random, uuid, os
from faker import Faker

fake = Faker("ru_RU")
random.seed(99)

CITIES_CFG = {
    "Сочи": {
        "count": 35,
        "lat": (43.55, 43.65), "lon": (39.70, 39.82),
        "areas": [
            ("Адлер",        "sea",       "ул. Ленина", 43.445, 39.920),
            ("Центр Сочи",   "center",    "ул. Навагинская", 43.600, 39.730),
            ("Хоста",        "sea",       "ул. Абрикосовая", 43.508, 39.848),
            ("Лазаревское",  "sea",       "ул. Победы", 43.908, 39.340),
        ],
        "metro_opts": ["ст. Сочи", "ост. Центральный рынок", "ост. Морской порт"],
        "has_beach": True, "has_ski": False,
    },
    "Красная Поляна": {
        "count": 35,
        "lat": (43.65, 43.73), "lon": (40.15, 40.28),
        "areas": [
            ("Роза Хутор",       "mountains", "ул. Эстосадок", 43.680, 40.295),
            ("Горки Город",      "mountains", "ул. Олимпийская", 43.670, 40.268),
            ("Красная Поляна",   "mountains", "ул. Заповедная", 43.680, 40.200),
        ],
        "metro_opts": ["Роза Хутор", "Эстосадок", "ост. Газпром Отель"],
        "has_beach": False, "has_ski": True,
    },
    "Санкт-Петербург": {
        "count": 40,
        "lat": (59.90, 59.98), "lon": (30.20, 30.42),
        "areas": [
            ("Центр",            "center",    "Невский пр-т", 59.933, 30.335),
            ("Петроградка",      "center",    "ул. Большая Пушкарская", 59.960, 30.307),
            ("Васильевский о-в", "sea",       "Университетская наб.", 59.940, 30.290),
            ("Адмиралтейский",   "center",    "пр-т Майорова", 59.925, 30.308),
        ],
        "metro_opts": ["Невский проспект", "Горьковская", "Василеостровская", "Сенная площадь"],
        "has_beach": False, "has_ski": False,
    },
    "Москва": {
        "count": 35,
        "lat": (55.70, 55.82), "lon": (37.50, 37.72),
        "areas": [
            ("Хамовники",        "center",    "ул. Льва Толстого", 55.730, 37.580),
            ("Замоскворечье",    "center",    "ул. Большая Ордынка", 55.737, 37.635),
            ("Пресня",           "center",    "ул. Красная Пресня", 55.760, 37.570),
            ("Деловой центр",    "center",    "Пресненская наб.", 55.749, 37.540),
        ],
        "metro_opts": ["Парк Культуры", "Третьяковская", "Деловой центр", "Новослободская"],
        "has_beach": False, "has_ski": False,
    },
    "Алтай": {
        "count": 25,
        "lat": (51.80, 52.10), "lon": (85.50, 86.05),
        "areas": [
            ("Чемал",            "nature",    "ул. Набережная", 51.400, 85.864),
            ("Манжерок",         "nature",    "ул. Центральная", 51.800, 85.820),
            ("Горно-Алтайск",    "mountains", "ул. Ленина", 51.959, 85.960),
        ],
        "metro_opts": ["ост. Чемал", "ост. Манжерок", "ост. Горно-Алтайск"],
        "has_beach": False, "has_ski": False,
    },
    "Казань": {
        "count": 30,
        "lat": (55.75, 55.84), "lon": (49.10, 49.22),
        "areas": [
            ("Кремль",           "center",    "ул. Кремлёвская", 55.796, 49.106),
            ("Набережная Волги", "sea",       "ул. Федосеевская", 55.800, 49.150),
            ("Старо-Татарская",  "center",    "ул. Татарстан", 55.785, 49.120),
        ],
        "metro_opts": ["Кремлёвская", "Площадь Тукая", "Суконная слобода"],
        "has_beach": False, "has_ski": False,
    },
}

PREFIXES = ["Гранд", "Уютный", "Лесной", "Горный", "Морской", "Премиум", "Камерный", "Панорама",
            "Альпийский", "Золотой", "Белый", "Северный", "Классик", "Бутик", "Арт"]
SUFFIXES = ["Отель", "Резорт", "Хаус", "Вилла", "Суит", "Инн", "Лодж", "Апарт", "Палас", "Спа"]

VIBES = {
    "mountains": [
        "Уютный горный отель с панорамными видами на заснеженные вершины и хвойные леса.",
        "Идеальная база для горнолыжного сезона — прямо у подъёмников, тёплый камин в лобби.",
        "Тихий шале-стиль в окружении гор: чистый воздух, терраса с видом на ущелье.",
    ],
    "sea": [
        "Свежий морской бриз и шум прибоя — пляж в шаговой доступности.",
        "Видовой отель на берегу: закаты над морем прямо с балкона номера.",
        "Уютный отель у набережной — лёгкая прогулка до моря и рыбных ресторанов.",
    ],
    "center": [
        "В самом сердце города — шаговая доступность музеев, ресторанов и транспортных узлов.",
        "Современный городской отель с отличной транспортной доступностью.",
        "Исторический район, атмосферные улочки, всё главное — рядом.",
    ],
    "nature": [
        "Эко-шале на берегу горной реки: тишина, природа и чистейший воздух Алтая.",
        "Погружение в природу без отказа от комфорта — тайга, река и звёздное небо.",
        "Глэмпинг-стиль в сосновом лесу — медитативный покой вдали от суеты.",
    ],
}

TAGS_MAP = {
    "mountains": ["вид_на_горы", "лыжи_сноуборд", "активный_отдых", "тишина"],
    "sea":        ["вид_на_море", "пляж", "романтика", "закат"],
    "center":     ["центр_города", "культура", "рестораны", "бизнес"],
    "nature":     ["эко", "природа", "тишина", "рыбалка"],
}

POS_REVIEWS = [
    "Великолепный тёплый бассейн с панорамой на горы",
    "Очень приветливый персонал, принесли лежанку для собаки",
    "Тихий район, спалось отлично",
    "Завтрак разнообразный, свежая выпечка каждое утро",
    "Чистейшие номера, хорошая звукоизоляция",
    "Отличный вид из окна, удобная парковка",
    "СПА-центр просто шикарный — сауна и хаммам",
    "Персонал говорит по-английски, очень помогли с экскурсиями",
]
NEG_REVIEWS = [
    "В выходные трудно найти место на парковке",
    "Завтрак хороший, но кофе из автомата",
    "Иногда слышно соседей сверху",
    "Wi-Fi в дальних номерах чуть слабее",
    "Мало розеток в номере",
    "Лифт медленный в часы пик",
]

VIEW_MAP = {"mountains": "mountain", "sea": "sea", "center": "city", "nature": "mountain"}

def gen_room_types(area_type, base_price):
    rooms = []
    view = VIEW_MAP.get(area_type, "city")
    configs = [
        ("std_single",   "Стандартный одноместный",              0.85, 1, 1, 2, "1 односпальная (Twin)", 18, False),
        ("std_double",   "Стандартный двухместный",               1.0,  2, 1, 3, "1 большая двуспальная (King Size)", 24, True),
        ("deluxe_dbl",   "Делюкс двухместный с видом",            1.4,  2, 1, 3, "1 King Size с панорамным окном", 32, True),
        ("family_suite", "Семейный сьют",                         1.9,  3, 2, 5, "1 двуспальная + 2 односпальных", 52, True),
        ("junior_suite", "Студия-люкс",                           2.2,  2, 0, 2, "1 King Size, гостиная зона", 45, True),
    ]
    chosen = random.sample(configs, k=random.randint(2, 4))
    for i, (rid, name, mult, ma, mc, mt, bed, sqm, balc) in enumerate(chosen):
        price = int(base_price * mult / 100) * 100
        rooms.append({
            "room_id": f"{rid}_{i+1}",
            "name": name,
            "price_per_night_rub": price,
            "capacity": {"max_adults": ma, "max_children": mc, "max_total_guests": mt},
            "bed_setup": bed,
            "area_sqm": sqm,
            "has_balcony": balc,
            "view": view,
            "breakfast_included": random.random() < 0.4,
            "available_rooms_count": random.randint(1, 6),
        })
    return rooms

def gen_hotel(idx, city, cfg):
    area_info = random.choice(cfg["areas"])
    area_name, area_type, street, base_lat, base_lon = area_info
    lat = round(base_lat + random.uniform(-0.02, 0.02), 6)
    lon = round(base_lon + random.uniform(-0.03, 0.03), 6)
    stars = random.choices([3, 4, 5], weights=[3, 4, 3])[0]
    base_price = {3: random.randint(3500, 8000), 4: random.randint(7000, 18000), 5: random.randint(15000, 40000)}[stars]

    pet_friendly  = random.random() < 0.40
    heated_pool   = random.random() < 0.35
    outdoor_pool  = random.random() < 0.25
    spa           = random.random() < 0.40
    sauna         = random.random() < 0.45
    fast_wifi     = random.random() < 0.90
    free_parking  = random.random() < 0.55
    restaurant    = random.random() < 0.65
    breakfast_av  = random.random() < 0.70
    gym           = random.random() < 0.50
    kids_club     = random.random() < 0.30
    coworking     = random.random() < 0.25
    ski_storage   = (area_type == "mountains") and (random.random() < 0.75)
    free_cancel   = random.random() < 0.60

    dist_beach    = random.randint(50, 800)   if cfg["has_beach"] else None
    dist_ski      = random.randint(50, 1200)  if cfg["has_ski"]   else None
    dist_center   = round(random.uniform(0.3, 12.0), 1)
    dist_transport= random.randint(50, 600)
    metro         = random.choice(cfg["metro_opts"])

    rating = round(random.uniform(3.2, 5.0), 1)
    reviews_count = random.randint(15, 2500)
    min_stay = random.choices([1, 2, 3], weights=[6, 3, 1])[0]

    room_types = gen_room_types(area_type, base_price)
    min_price = min(r["price_per_night_rub"] for r in room_types)

    tags = list(TAGS_MAP[area_type])
    if pet_friendly:  tags.append("дог-френдли")
    if spa:           tags.append("спа_релакс")
    if kids_club:     tags.append("для_семей")
    if coworking:     tags.append("работа_коворкинг")
    if romantic := (random.random() < 0.3): tags.append("романтика")
    tags = list(set(tags))

    amenity_parts = []
    if heated_pool:  amenity_parts.append("подогреваемый бассейн")
    if spa:          amenity_parts.append("спа-центр")
    if sauna:        amenity_parts.append("сауна")
    if restaurant:   amenity_parts.append("ресторан")
    if gym:          amenity_parts.append("фитнес-зал")
    if pet_friendly: amenity_parts.append("размещение питомцев")
    if free_parking: amenity_parts.append("бесплатная парковка")
    if kids_club:    amenity_parts.append("детский клуб")

    vibe = random.choice(VIBES[area_type])
    description = (
        f"{vibe} "
        f"Расположен в районе «{area_name}» ({city}), адрес: {street}, {random.randint(1,120)}. "
        f"Рейтинг гостей: {rating}/5.0 на основе {reviews_count} отзывов. "
        f"Звёздность: {'★' * stars}. "
        f"Цены от {min_price:,} ₽/ночь. "
        + (f"До моря {dist_beach} м. " if dist_beach else "")
        + (f"До подъёмника {dist_ski} м. " if dist_ski else "")
        + f"До центра {dist_center} км. "
        + (f"Удобства: {', '.join(amenity_parts)}. " if amenity_parts else "")
        + f"Теги: {', '.join(tags)}."
    )

    name = f"{random.choice(PREFIXES)} {random.choice(SUFFIXES)} {idx}"

    return {
        "id": str(uuid.uuid4()),
        "name": name,
        "city": city,
        "stars": stars,
        "area": area_name,
        "area_type": area_type,
        "address": f"{street}, {random.randint(1, 120)}",
        "coordinates": {"lat": lat, "lon": lon},
        "min_price_per_night_rub": min_price,
        "rating": rating,
        "reviews_count": reviews_count,
        "location_details": {
            "distance_to_center_km": dist_center,
            "distance_to_beach_m": dist_beach,
            "distance_to_ski_lift_m": dist_ski,
            "nearest_metro_or_station": metro,
            "distance_to_transport_m": dist_transport,
        },
        "amenities": {
            "pet_friendly": pet_friendly, "heated_pool": heated_pool,
            "outdoor_pool": outdoor_pool, "spa": spa, "sauna": sauna,
            "fast_wifi": fast_wifi, "free_parking": free_parking,
            "paid_parking": not free_parking and random.random() < 0.5,
            "restaurant_on_site": restaurant,
            "breakfast_included_available": breakfast_av,
            "gym": gym, "kids_club": kids_club,
            "coworking_zone": coworking, "ski_storage": ski_storage,
        },
        "room_types": room_types,
        "policies": {
            "check_in_time": random.choice(["13:00", "14:00", "15:00"]),
            "check_out_time": random.choice(["11:00", "12:00"]),
            "early_check_in_available": random.random() < 0.5,
            "late_check_out_available": random.random() < 0.4,
            "min_stay_nights": min_stay,
            "cancellation": {
                "is_free_cancellation": free_cancel,
                "free_cancellation_deadline_hours": 48 if free_cancel else 0,
                "penalty_rub": 0 if free_cancel else min_price,
            },
            "pets": {
                "allowed": pet_friendly,
                "max_pets": random.choice([1, 2]) if pet_friendly else 0,
                "max_weight_kg": random.choice([5, 10, 15, 20]) if pet_friendly else 0,
                "deposit_rub": random.randint(1000, 5000) if pet_friendly else 0,
                "daily_fee_rub": random.choice([0, 300, 500, 700]) if pet_friendly else 0,
                "allowed_types": random.choice([["dogs","cats"], ["dogs"], ["cats"], ["dogs","cats","birds"]]) if pet_friendly else [],
                "restrictions_note": f"Питомцы до {random.choice([5,10,15,20])} кг с ветпаспортом" if pet_friendly else "Питомцы не разрешены",
            },
            "children": {
                "allowed": True,
                "free_child_age_limit": random.choice([3, 5, 6, 7]),
                "extra_bed_price_rub": random.randint(800, 2500),
            },
        },
        "tags": tags,
        "reviews_summary": {
            "score_cleanliness": round(random.uniform(3.5, 5.0), 1),
            "score_location":    round(random.uniform(3.5, 5.0), 1),
            "score_service":     round(random.uniform(3.2, 5.0), 1),
            "positive_highlights": random.sample(POS_REVIEWS, k=random.randint(2, 4)),
            "negative_highlights": random.sample(NEG_REVIEWS,  k=random.randint(1, 3)),
        },
        "description": description,
        "image_urls": [
            f"https://images.ostrovok.ru/mock/{uuid.uuid4().hex[:8]}_exterior.jpg",
            f"https://images.ostrovok.ru/mock/{uuid.uuid4().hex[:8]}_room.jpg",
            f"https://images.ostrovok.ru/mock/{uuid.uuid4().hex[:8]}_pool_spa.jpg",
            f"https://images.ostrovok.ru/mock/{uuid.uuid4().hex[:8]}_breakfast.jpg",
        ],
    }

hotels = []
idx = 1
for city, cfg in CITIES_CFG.items():
    for _ in range(cfg["count"]):
        hotels.append(gen_hotel(idx, city, cfg))
        idx += 1

os.makedirs("./outputs", exist_ok=True)
with open("./outputs/hotels_dataset.json", "w", encoding="utf-8") as f:
    json.dump(hotels, f, ensure_ascii=False, indent=2)

print(f"✅ Датасет: {len(hotels)} отелей")
for city in CITIES_CFG:
    c = sum(1 for h in hotels if h["city"] == city)
    print(f"   {city}: {c}")
print(f"   Pet-friendly: {sum(h['amenities']['pet_friendly'] for h in hotels)}")
print(f"   Подогрев. бассейн: {sum(h['amenities']['heated_pool'] for h in hotels)}")
print(f"   СПА: {sum(h['amenities']['spa'] for h in hotels)}")
print(f"   Бесплатная отмена: {sum(h['policies']['cancellation']['is_free_cancellation'] for h in hotels)}")


✅ Датасет: 200 отелей
   Сочи: 35
   Красная Поляна: 35
   Санкт-Петербург: 40
   Москва: 35
   Алтай: 25
   Казань: 30
   Pet-friendly: 70
   Подогрев. бассейн: 61
   СПА: 82
   Бесплатная отмена: 115


## 7. Пересоздание ChromaDB-индекса (расширенные метаданные)


In [13]:

import chromadb, shutil, os
from sentence_transformers import SentenceTransformer

# Используем свежий путь чтобы избежать конфликта схем со старым DB
DB_PATH = "./outputs/chroma_db_v2"
if os.path.exists(DB_PATH):
    shutil.rmtree(DB_PATH)
os.makedirs(DB_PATH, exist_ok=True)

model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
chroma_client = chromadb.PersistentClient(path=DB_PATH)
collection = chroma_client.create_collection("hotels", metadata={"hnsw:space": "cosine"})

descriptions = [h["description"] for h in hotels]
print("Генерация эмбеддингов для 200 отелей...")
embeddings = model.encode(descriptions, batch_size=32, show_progress_bar=True).tolist()

ids = [h["id"] for h in hotels]
metadatas = []
for h in hotels:
    loc = h["location_details"]
    metadatas.append({
        "hotel_id":              h["id"],
        "name":                  h["name"],
        "city":                  h["city"],
        "area_type":             h["area_type"],
        "stars":                 h["stars"],
        "min_price_rub":         h["min_price_per_night_rub"],
        "rating":                h["rating"],
        "reviews_count":         h["reviews_count"],
        "pet_friendly":          str(h["amenities"]["pet_friendly"]),
        "heated_pool":           str(h["amenities"]["heated_pool"]),
        "spa":                   str(h["amenities"]["spa"]),
        "breakfast_included":    str(h["amenities"]["breakfast_included_available"]),
        "free_cancellation":     str(h["policies"]["cancellation"]["is_free_cancellation"]),
        "min_stay_nights":       h["policies"]["min_stay_nights"],
        "lat":                   h["coordinates"]["lat"],
        "lon":                   h["coordinates"]["lon"],
        "distance_to_ski_lift_m": loc["distance_to_ski_lift_m"] if loc["distance_to_ski_lift_m"] is not None else 99999,
        "distance_to_beach_m":   loc["distance_to_beach_m"] if loc["distance_to_beach_m"] is not None else 99999,
    })

collection.add(ids=ids, embeddings=embeddings, documents=descriptions, metadatas=metadatas)
print(f"✅ ChromaDB v2: {collection.count()} документов в индексе → {DB_PATH}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Генерация эмбеддингов для 200 отелей...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

✅ ChromaDB v2: 200 документов в индексе → ./outputs/chroma_db_v2


## 8. Обновлённый модуль `hotel_search.py` — расширенные фильтры + `HotelDetailResult`


In [16]:

# Фикс: убираем `from __future__ import annotations` — конфликт с Pydantic v2 при динамической загрузке
hotel_search_v2 = '''"""
hotel_search.py — v2
"""
import json, os, asyncio
from typing import Optional, List, Any, Dict
from pydantic import BaseModel, Field
import chromadb
from sentence_transformers import SentenceTransformer

class HotelFilterSchema(BaseModel):
    city: Optional[str] = Field(None)
    area_type: Optional[str] = Field(None)
    stars: Optional[int] = Field(None)
    max_price_per_night_rub: Optional[int] = Field(None)
    min_rating: Optional[float] = Field(None)
    pet_friendly: Optional[bool] = Field(None)
    heated_pool: Optional[bool] = Field(None)
    spa: Optional[bool] = Field(None)
    breakfast_included: Optional[bool] = Field(None)
    free_cancellation: Optional[bool] = Field(None)
    max_distance_to_beach_m: Optional[int] = Field(None)
    max_distance_to_ski_lift_m: Optional[int] = Field(None)
    guests_count: Optional[int] = Field(None)
    top_k: int = Field(5)

class HotelDetailResult(BaseModel):
    id: str
    name: str
    city: str
    stars: int
    area: str
    area_type: str
    address: str
    min_price_per_night_rub: int
    rating: float
    reviews_count: int
    location_details: Dict[str, Any]
    amenities: Dict[str, Any]
    room_types: List[Dict[str, Any]]
    policies: Dict[str, Any]
    tags: List[str]
    reviews_summary: Dict[str, Any]
    description: str
    image_urls: List[str]
    score: float
    suitable_rooms: List[Dict[str, Any]] = Field(default_factory=list)

_model = None
_collection = None
_hotels_index: Dict[str, Dict] = {}

def _resolve_db_path():
    for p in ["./outputs/chroma_db_v2", "./outputs/chroma_db"]:
        if os.path.exists(p):
            return p
    raise FileNotFoundError("ChromaDB не найдена.")

def _resolve_dataset_path():
    for p in ["./outputs/hotels_dataset.json", "./hotels_dataset.json"]:
        if os.path.exists(p):
            return p
    raise FileNotFoundError("hotels_dataset.json не найден.")

def _init():
    global _model, _collection, _hotels_index
    if _model is None:
        _model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
    if _collection is None:
        client = chromadb.PersistentClient(path=_resolve_db_path())
        _collection = client.get_collection("hotels")
    if not _hotels_index:
        with open(_resolve_dataset_path(), encoding="utf-8") as f:
            _hotels_index = {h["id"]: h for h in json.load(f)}

async def search_hotels(filters: HotelFilterSchema, semantic_query: str) -> List[HotelDetailResult]:
    _init()
    clauses = []
    if filters.city:                          clauses.append({"city": {"$eq": filters.city}})
    if filters.area_type:                     clauses.append({"area_type": {"$eq": filters.area_type}})
    if filters.stars is not None:             clauses.append({"stars": {"$gte": filters.stars}})
    if filters.max_price_per_night_rub:       clauses.append({"min_price_rub": {"$lte": filters.max_price_per_night_rub}})
    if filters.min_rating:                    clauses.append({"rating": {"$gte": filters.min_rating}})
    if filters.pet_friendly is not None:      clauses.append({"pet_friendly": {"$eq": str(filters.pet_friendly)}})
    if filters.heated_pool is not None:       clauses.append({"heated_pool": {"$eq": str(filters.heated_pool)}})
    if filters.spa is not None:               clauses.append({"spa": {"$eq": str(filters.spa)}})
    if filters.breakfast_included is not None:clauses.append({"breakfast_included": {"$eq": str(filters.breakfast_included)}})
    if filters.free_cancellation is not None: clauses.append({"free_cancellation": {"$eq": str(filters.free_cancellation)}})
    if filters.max_distance_to_beach_m:       clauses.append({"distance_to_beach_m": {"$lte": filters.max_distance_to_beach_m}})
    if filters.max_distance_to_ski_lift_m:    clauses.append({"distance_to_ski_lift_m": {"$lte": filters.max_distance_to_ski_lift_m}})

    where = {"$and": clauses} if len(clauses) > 1 else (clauses[0] if clauses else None)
    query_emb = await asyncio.to_thread(_model.encode, [semantic_query])
    kwargs = dict(query_embeddings=query_emb.tolist(),
                  n_results=min(filters.top_k, _collection.count()),
                  include=["metadatas", "documents", "distances"])
    if where:
        kwargs["where"] = where
    res = _collection.query(**kwargs)

    out = []
    for i, (meta, doc, dist) in enumerate(zip(res["metadatas"][0], res["documents"][0], res["distances"][0])):
        hotel_id = res["ids"][0][i]
        full = _hotels_index.get(hotel_id)
        if not full:
            continue
        suitable = [r for r in full.get("room_types", [])
                    if r["capacity"]["max_total_guests"] >= (filters.guests_count or 1)]
        out.append(HotelDetailResult(
            id=hotel_id, name=full["name"], city=full["city"], stars=full["stars"],
            area=full["area"], area_type=full["area_type"], address=full["address"],
            min_price_per_night_rub=full["min_price_per_night_rub"],
            rating=full["rating"], reviews_count=full["reviews_count"],
            location_details=full["location_details"], amenities=full["amenities"],
            room_types=full["room_types"], policies=full["policies"], tags=full["tags"],
            reviews_summary=full["reviews_summary"], description=doc,
            image_urls=full.get("image_urls", []),
            score=round(1 - dist, 4), suitable_rooms=suitable,
        ))
    return out
'''

with open("./hotel_search.py", "w", encoding="utf-8") as f:
    f.write(hotel_search_v2)
print("✅ hotel_search.py v2 (fixed) сохранён")


✅ hotel_search.py v2 (fixed) сохранён


## 9. End-to-end тест: «Семья в Красной Поляне, бассейн, до подъёмника, ≤20 000₽/ночь»


In [17]:

import importlib.util, asyncio, sys

# Убираем кэш модуля если был загружен ранее
if "hotel_search" in sys.modules:
    del sys.modules["hotel_search"]

spec = importlib.util.spec_from_file_location("hotel_search", "./hotel_search.py")
hs = importlib.util.module_from_spec(spec)
spec.loader.exec_module(hs)

filters = hs.HotelFilterSchema(
    city="Красная Поляна",
    heated_pool=True,
    max_price_per_night_rub=20000,
    max_distance_to_ski_lift_m=1200,
    guests_count=3,
    top_k=3,
)

results = await hs.search_hotels(
    filters=filters,
    semantic_query="семейный отдых в горах, подогреваемый бассейн, рядом с подъёмником, для двоих с ребёнком",
)

print("=" * 70)
print("ЗАПРОС: Семья (2+1) · Красная Поляна · бассейн · ≤20 000₽ · ≤1200 м до подъёмника")
print("=" * 70)

for rank, h in enumerate(results, 1):
    pol   = h.policies
    canc  = pol["cancellation"]
    child = pol["children"]
    loc   = h.location_details
    best_room = sorted(h.suitable_rooms, key=lambda r: r["price_per_night_rub"])[0] if h.suitable_rooms else None

    print(f"\n{'─'*70}")
    print(f"  #{rank}  {h.name}  ({'★'*h.stars})  │  Рейтинг: {h.rating}/5  │  Отзывов: {h.reviews_count}")
    print(f"       📍 {h.city}, {h.area} — {h.address}")
    print(f"       🎿 До подъёмника: {loc['distance_to_ski_lift_m']} м  │  До центра: {loc['distance_to_center_km']} км")
    print(f"       🏊 Бассейн: {'✅' if h.amenities['heated_pool'] else '❌'}  "
          f"СПА: {'✅' if h.amenities['spa'] else '❌'}  "
          f"Парковка: {'✅' if h.amenities['free_parking'] else '❌'}")
    print(f"       🏷  Теги: {', '.join(h.tags)}")

    if best_room:
        bkfast = "✅ включён" if best_room["breakfast_included"] else "❌ не включён"
        print(f"\n  💤 НОМЕР: {best_room['name']}  ({best_room['area_sqm']} м², вид: {best_room['view']})")
        print(f"       Кровати: {best_room['bed_setup']}  │  Балкон: {'✅' if best_room['has_balcony'] else '❌'}")
        print(f"       Вместимость: взрослых {best_room['capacity']['max_adults']}, детей {best_room['capacity']['max_children']}")
        print(f"       Завтрак: {bkfast}")
        print(f"       💰 Цена: {best_room['price_per_night_rub']:,} ₽/ночь  │  3 ночи: {best_room['price_per_night_rub']*3:,} ₽")

    cancel_str = (f"✅ Бесплатная отмена за {canc['free_cancellation_deadline_hours']} ч"
                  if canc["is_free_cancellation"] else f"❌ Штраф: {canc['penalty_rub']:,} ₽")
    print(f"\n  📋 Заезд: {pol['check_in_time']}  │  Выезд: {pol['check_out_time']}  │  "
          f"Мин. срок: {pol['min_stay_nights']} н.  │  Отмена: {cancel_str}")
    print(f"       Дети до {child['free_child_age_limit']} лет — бесплатно  │  Доп. кровать: {child['extra_bed_price_rub']:,} ₽")

    rev = h.reviews_summary
    print(f"\n  ⭐ Чистота {rev['score_cleanliness']} │ Локация {rev['score_location']} │ Сервис {rev['score_service']}")
    print(f"     👍 {rev['positive_highlights'][0]}")
    if rev.get("negative_highlights"):
        print(f"     👎 {rev['negative_highlights'][0]}")
    print(f"     Score: {h.score}")

print("\n" + "=" * 70)
print(f"✅ Тест пройден: найдено {len(results)} отелей")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

ЗАПРОС: Семья (2+1) · Красная Поляна · бассейн · ≤20 000₽ · ≤1200 м до подъёмника

──────────────────────────────────────────────────────────────────────
  #1  Гранд Апарт 62  (★★★)  │  Рейтинг: 4.3/5  │  Отзывов: 842
       📍 Красная Поляна, Красная Поляна — ул. Заповедная, 59
       🎿 До подъёмника: 515 м  │  До центра: 7.3 км
       🏊 Бассейн: ✅  СПА: ✅  Парковка: ❌
       🏷  Теги: тишина, активный_отдых, вид_на_горы, спа_релакс, для_семей, лыжи_сноуборд

  💤 НОМЕР: Стандартный двухместный  (24 м², вид: mountain)
       Кровати: 1 большая двуспальная (King Size)  │  Балкон: ✅
       Вместимость: взрослых 2, детей 1
       Завтрак: ✅ включён
       💰 Цена: 6,500 ₽/ночь  │  3 ночи: 19,500 ₽

  📋 Заезд: 14:00  │  Выезд: 11:00  │  Мин. срок: 3 н.  │  Отмена: ❌ Штраф: 6,500 ₽
       Дети до 7 лет — бесплатно  │  Доп. кровать: 998 ₽

  ⭐ Чистота 3.9 │ Локация 3.8 │ Сервис 3.5
     👍 Персонал говорит по-английски, очень помогли с экскурсиями
     👎 Завтрак хороший, но кофе из автомата
    